# Intro to Decorators

For this notebook, we recommend splitting the VS Code editor. You can keep this notebook open in one panel and the related files open in another while you work.

This notebook lives in `05-intro-to-decorators/`:

<pre>
05-intro-to-decorators/
├── 05-intro-to-decorators.ipynb
├── src/
│   └── unit_test_examples/
│       ├── __init__.py
│       ├── type_check_solution.py
│       └── type_check.py
└── tests/
    ├── test_type_check_solution.py
    └── test_type_check.py
</pre>

[Decorators](https://docs.python.org/3/glossary.html#term-decorator) often feel abstract the first time you see them because several functions are involved at once. The key idea in this notebook is that decorators are built from ordinary nested functions: one function captures values, another wraps the original callable, and the wrapper decides what happens before or after the original function runs.

A useful mental model is that a decorator does not magically change a function. It returns a new callable that stands in front of the original one. Once that model is clear, decorators stop feeling like special syntax and start looking like a structured function-composition pattern.

This notebook focuses on three ideas:

* nested functions and closures
* one basic decorator pattern
* one decorator factory pattern, which leads directly to `type_check`

**What to expect:** the pytest implementation-target check for `type_check.py` should fail until the decorator is implemented. The reference check for `type_check_solution.py` should pass.

> All commands in this notebook assume you are inside `05-intro-to-decorators/`.


In [ ]:
import sys
from pathlib import Path

for candidate in (
    Path.cwd().resolve(),
    Path.cwd().resolve() / "05-intro-to-decorators",
):
    if (candidate / "src").exists():
        NOTEBOOK_ROOT = candidate
        break
else:
    NOTEBOOK_ROOT = Path.cwd().resolve()

if str(NOTEBOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(NOTEBOOK_ROOT))

## Decorator Flow

```mermaid
flowchart LR
    A["type_check(correct_type)"] --> B["decorator(function)"]
    B --> C["wrapper(*args, **kwargs)"]
    C --> D["original function"]
    D --> E["returned value"]
    E --> C
```

The important thing to notice is that each layer has a different job. The outer factory captures configuration, the decorator receives the original function, and the wrapper runs at call time and decides what to do with the returned value.

This separation of responsibilities is what makes decorators flexible. Once you understand which layer owns configuration and which layer owns runtime behavior, parameterized decorators become much easier to reason about.


## Closures

Decorators rely on nested functions. A nested function can capture values from the outer scope and keep using them later.

That stored access to outer variables is what lets a decorator remember configuration such as `correct_type` even after the outer function has already finished running.


In [ ]:
def outer(x):
    def inner(y):
        return x + y

    return inner


add_five = outer(5)
add_five(6)

`outer(5)` returns `inner`, and that returned function keeps access to `x`. This closure pattern is the same mechanism decorators use when they wrap another function.

The important point is that the returned function is not copying the value in a special way. It simply keeps a reference to the surrounding scope, which is why closures are the foundation for decorators with configuration.


## Basic Decorator

A basic decorator takes a function, wraps extra behavior around it, and returns a new callable.

The key idea is that the decorated function name now points to the wrapper, not directly to the original function. The wrapper can therefore run code before the original call, after it, or even inspect and modify the returned result.


In [ ]:
def make_pretty(function):
    def wrapper():
        print("I got decorated")
        function()

    return wrapper


@make_pretty
def hello_world():
    print("Hello World")


hello_world()

The important connection is that `make_pretty()` returns `wrapper()`, and `wrapper()` decides when the original function is called.

That is the central mental model for decorators: you are not editing the original function body in place. You are building another function that controls how and when the original function is executed.


## Decorator Factory

Sometimes the decorator itself needs configuration. In that case you use three layers:

* factory
* decorator
* wrapper


In [ ]:
def multiply_by(multiplier):
    def decorator(function):
        def wrapper(*args, **kwargs):
            result = function(*args, **kwargs)
            return result * multiplier

        return wrapper

    return decorator


@multiply_by(3)
def base_value():
    return 4


base_value()

This factory pattern is exactly what `type_check(correct_type)` needs: the outer function captures the expected type once, then the wrapper validates each returned value.

That is why parameterized decorators use an extra layer. Without the outer factory, there would be nowhere to store configuration such as the expected return type.


## type_check Implementation Target

Implement `type_check(correct_type)` in `src/unit_test_examples/type_check.py`.

Required behavior:

* if the wrapped function returns a value of the expected type, return that value unchanged
* if the return type does not match, print `Bad Type` and return `None`

Example contract:

```python
@type_check(int)
def times2(num):
    return num * 2
```


1. Open `src/unit_test_examples/type_check.py`.
2. Implement the factory, decorator, and wrapper layers.
3. Call the wrapped function inside the wrapper and inspect its returned value.
4. Run `../.venv/bin/python -m pytest -q tests/test_type_check.py`.
5. Expect that command to fail until the implementation is complete, then pass once the behavior matches the contract.


In [ ]:
# @TODO Implementation Target: Implement type_check decorator.
# Objective: Enforce the decorated function return type and print "Bad Type" on mismatch.
# Edit files:
# - src/unit_test_examples/type_check.py
# Validate with:
# - ../.venv/bin/python -m pytest -q tests/test_type_check.py
# Solution:
# - src/unit_test_examples/type_check_solution.py


<details>
  <summary>Solution</summary>

This pattern uses three layers: factory (`type_check`) -> decorator -> wrapper.

```python
# src/unit_test_examples/type_check_solution.py

def type_check(correct_type):
    def decorator(function):
        def wrapper(*args, **kwargs):
            result = function(*args, **kwargs)
            if isinstance(result, correct_type):
                return result
            print("Bad Type")
            return None

        return wrapper

    return decorator
```
</details>


## Running Tests

From inside `05-intro-to-decorators/`:

##### Validation command:

```zsh
../.venv/bin/python -m pytest -q tests/test_type_check.py
```

**What to expect:** `tests/test_type_check.py` should fail until `src/unit_test_examples/type_check.py` is implemented

##### Reference check:

```zsh
../.venv/bin/python -m pytest -q tests/test_type_check_solution.py
```


##### Run all module tests:

```zsh
../.venv/bin/python -m pytest -q tests
```

Because this notebook intentionally includes an unfinished implementation target, the full test run will fail until `type_check.py` is completed.
